# Module 10 — Vision Transformers

Transformers replaced convolutions as the dominant vision architecture.
We build attention and ViT from scratch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from attention import MultiHeadAttention, PatchEmbed, TransformerBlock, SwinBlock, window_partition
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Self-Attention from Scratch

Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V

In [ ]:
import math
def scaled_dot_product_attention(q, k, v):
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    return weights @ v, weights

# Demo: 4 tokens, d=8
B, N, D = 1, 4, 8
q = k = v = torch.randn(B, N, D)
out, weights = scaled_dot_product_attention(q, k, v)
print('Output shape:', out.shape)
print('Attention weights (row sum = 1):', weights[0].sum(dim=-1))

## 2. Patch Embedding

In [ ]:
patch_embed = PatchEmbed(img_size=224, patch_size=16, in_channels=3, embed_dim=768)
x = torch.randn(2, 3, 224, 224)
tokens = patch_embed(x)
print(f'Image {x.shape} → Tokens {tokens.shape}')
print(f'Num patches: {patch_embed.num_patches}')  # 196

# Visualize patch grid
fig, ax = plt.subplots(1,1,figsize=(5,5))
patch_vis = np.ones((224,224,3))
for i in range(0,224,16):
    patch_vis[i,:] = 0.7
    patch_vis[:,i] = 0.7
ax.imshow(patch_vis)
ax.set_title('224x224 image → 14x14 = 196 patches (16x16 each)')
ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Mini ViT

Building a complete ViT-Tiny encoder.

In [ ]:
class ViTTiny(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=1000,
                 d_model=192, n_heads=3, n_layers=12, mlp_ratio=4.0):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, 3, d_model)
        N = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, N + 1, d_model))
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, mlp_ratio) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.blocks(x)
        x = self.norm(x[:, 0])  # CLS token
        return self.head(x)

vit = ViTTiny()
params = sum(p.numel() for p in vit.parameters())/1e6
print(f'ViT-Tiny parameters: {params:.1f}M')
x = torch.randn(1, 3, 224, 224)
y = vit(x)
print('Output shape:', y.shape)

## 4. Swin Transformer Windows

In [ ]:
# Window partition demo
H = W = 56  # feature map spatial size
C = 96
x_feat = torch.randn(1, H, W, C)
windows = window_partition(x_feat, window_size=7)
print(f'Feature map {(1,H,W,C)} → {windows.shape} windows')
# (1, 56, 56, 96) → (64, 7, 7, 96) — 8x8 = 64 windows

# Swin block
swin = SwinBlock(d_model=96, n_heads=3, window_size=7, shift_size=0)
x_tokens = x_feat.view(1, H*W, C)
out = swin(x_tokens)
print('Swin output:', out.shape)

## Exercise — Relative Position Bias

In Swin Transformer, attention scores are augmented with learnable *relative position bias*:

```
Attn(Q,K,V) = softmax((QKᵀ + B) / √d_k) V
```

where B is a (W_s², W_s²) bias table indexed by relative token positions.

**Task:** implement `RelativePositionBias(window_size)` that returns a (Ws², Ws²) bias matrix.

In [ ]:
### EXERCISE
class RelativePositionBias(nn.Module):
    def __init__(self, window_size=7):
        super().__init__()
        # TODO: define parameter table and index
        raise NotImplementedError
    def forward(self):
        # Returns (Ws*Ws, Ws*Ws) bias
        raise NotImplementedError